In [ ]:
import os

# Core LSD imports
from sclsd.core import LSDConfig
from sclsd.train import LSD
from sclsd.plotting import plot_random_walks
from sclsd.utils import set_all_seeds
from sclsd.utils.seed import clear_pyro_state
from sclsd.plotting import plot_random_walks
from sclsd.preprocessing import get_prior_transition


# Scientific computing
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# Deep learning
import torch


# Single-cell analysis
import scanpy as sc

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set random seeds for reproducibility
SEED = 42
set_all_seeds(SEED)

In [ ]:
adata = sc.read("../../Zenodo/Zebrafish_Pigment/preprocessed_adata.h5ad")

# Add pseudotime, aligned by cell ID
phylogeny = {
    "neural crest + pigment cell progenitor":      ['xanthophore', 'melanophore'],                   
    "xanthophore":            [],                 
    "melanophore":              [],                  

}
sc.pl.umap(adata, color = ["prior_pseudotime", "clusters"])

In [ ]:
# Configure optimization hyperparameters
cfg = LSDConfig()
cfg.optimizer.adam.lr = 2e-3
cfg.walks.path_len = 32
cfg.walks.num_walks = 8192
cfg.walks.batch_size = 1024


# Initialize LSD model
lsd = LSD(
    adata,
    cfg,
    device=device,

)
lsd.set_phylogeny(phylogeny, cluster_key="clusters")
lsd.set_prior_transition(prior_time_key= "prior_pseudotime")
lsd.prepare_walks()

In [ ]:
# Define model directory based on hyperparameters
model_dir = "./model"

# Clear parameter store for fresh training
clear_pyro_state()
# Create save directory
os.makedirs(model_dir, exist_ok=True)
num_epochs = 60     # Total training epochs
# Train the model
lsd.train(
    num_epochs=num_epochs,
    save_dir=model_dir,
    random_state=SEED,
    save_interval= 20,
)

In [ ]:
#extract LSD outputs (transition matrix, potantial, entropy, pseudotime)
final_adata = lsd.get_adata()

#predict cell fates
dyn_adata = lsd.get_cell_fates(final_adata , time_range = 20, cluster_key = "clusters", batch_size = 2048)

In [ ]:
#plot lsd outputs
cols = ["potential", "lsd_pseudotime", "fate", "entropy"]

print("="* 60)
print("plotting LSD outputs on force directed projection")
print("="* 60)
sc.pl.embedding(dyn_adata, color = cols, basis = "X_umap", ncols = 2)

print("="* 60)
print("plotting LSD outputs on diffrentiation state projection")
print("="* 60)
sc.pl.embedding(dyn_adata, color = cols, basis = "X_diff_state", ncols = 2)

print("="* 60)
print("plotting LSD streamlines on force directed projection")
print("="* 60)
lsd.stream_lines(embedding= "X_umap", size = 60)
